# 02 — SP Type Classifier

This notebook trains CPP-based one-vs-rest classifiers for signal peptide type prediction.

The pipeline follows the cleaned terminology used in this repository:

- `pre_cs`: sequence region before the annotated cleavage site
- `post_cs`: sequence region after the annotated cleavage site
- `cs_position`: annotated cleavage-site position

The workflow is:

1. Load the parsed SignalP 6.0 dataset
2. Keep true SP sequences only
3. Split SP sequences into train/test sets stratified by SP type
4. Train one-vs-rest CPP classifiers for each SP type
5. Convert OVR probabilities into multiclass SP type predictions
6. Optionally refine difficult pairwise decisions:
   - `LIPO` vs `TATLIPO`
   - `SP` vs `TAT`

## 1. Imports

In [ ]:
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import aaanalysis as aa
import aaanalysis.utils as ut

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

try:
    from lightgbm import LGBMClassifier
except Exception:
    LGBMClassifier = None

try:
    from catboost import CatBoostClassifier
except Exception:
    CatBoostClassifier = None

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


aa.options["verbose"] = False
aa.options["random_state"] = 42

## 2. Paths and configuration

This notebook uses `dataset_parsed.csv` created by `scripts/01_preprocess_signalp6.py`.

Filtering and balancing are performed explicitly inside this notebook to make the experimental setup clear.

In [ ]:
PROJECT_ROOT = Path("..").resolve()

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_parsed.csv"

FEATURE_DIR = PROJECT_ROOT / "features"
MODEL_DIR = PROJECT_ROOT / "models"
PLOT_DIR = PROJECT_ROOT / "plots"
METRIC_DIR = PROJECT_ROOT / "results" / "metrics"

for d in [
    FEATURE_DIR / "metadata",
    FEATURE_DIR / "matrices",
    FEATURE_DIR / "indices",
    MODEL_DIR,
    PLOT_DIR,
    METRIC_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_RUNS = 10

MIN_PRE_CS_LEN = 5
POST_CS_LEN = 15
PRE_CS_LEN_FOR_PLOTS = 24
MAX_COR = 0.5

SP_TYPES = ["SP", "LIPO", "TAT", "TATLIPO", "PILIN"]
OVR_NEG_POS_RATIO = 8

## 3. Load parsed dataset

Required columns:

- `entry`
- `kingdom`
- `sp_type`
- `sequence`
- `annotation`
- `label_binary`
- `cs_position`
- `pre_cs`
- `post_cs`
- `pre_cs_len`
- `post_cs_len`

In [ ]:
def load_parsed_dataset(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)

    required = [
        "entry",
        "kingdom",
        "sp_type",
        "sequence",
        "annotation",
        "label_binary",
        "cs_position",
        "pre_cs",
        "post_cs",
        "pre_cs_len",
        "post_cs_len",
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df.copy()
    df["label_binary"] = df["label_binary"].astype(int)
    df["pre_cs"] = df["pre_cs"].astype(str)
    df["post_cs"] = df["post_cs"].astype(str)

    return df


df_seq = load_parsed_dataset(DATA_PATH)

# Same biological coverage choice as the binary classifier notebook.
df_seq = df_seq[df_seq["pre_cs_len"] >= MIN_PRE_CS_LEN].copy().reset_index(drop=True)

print(df_seq.shape)
print(df_seq["label_binary"].value_counts())
print(df_seq["sp_type"].value_counts())

## 4. Build SP-only train/test sets

The SP type classifier is trained only on true SP sequences. The train/test split is stratified by `sp_type` to keep rare classes such as `PILIN` and `TATLIPO` represented in both sets.

In [ ]:
df_sp = df_seq[df_seq["label_binary"] == 1].copy().reset_index(drop=True)
df_sp = df_sp[df_sp["sp_type"].isin(SP_TYPES)].copy().reset_index(drop=True)

train_sp_df, test_sp_df = train_test_split(
    df_sp,
    test_size=TEST_SIZE,
    stratify=df_sp["sp_type"],
    random_state=RANDOM_STATE,
)

train_sp_df = train_sp_df.reset_index(drop=True)
test_sp_df = test_sp_df.reset_index(drop=True)

print("Train SP type counts:")
print(train_sp_df["sp_type"].value_counts())

print("\nTest SP type counts:")
print(test_sp_df["sp_type"].value_counts())

## 5. Load AA scales and define CPP adapter

The cleaned dataset uses `pre_cs` and `post_cs`, but the AAanalysis CPP implementation expects internal part names such as `tmd` and `jmd_c`.

Therefore, the columns are mapped only at the CPP interface:

- `tmd` = `pre_cs`
- `jmd_c` = `post_cs`

In [ ]:
df_scales = aa.load_scales()

# Redundancy-reduced scale set, matching the style of the original experiments.
aac = aa.AAclust()
X_scales = np.array(df_scales).T
selected_scales = aac.fit(
    X_scales,
    names=list(df_scales),
    n_clusters=100,
).medoid_names_
df_scales = df_scales[selected_scales]

sf = aa.SequenceFeature()

SPLIT_KWS = {
    "Segment": {
        "n_split_min": 1,
        "n_split_max": 5,
    },
    "Pattern": {
        "steps": [3, 4],
        "n_min": 1,
        "n_max": 2,
        "len_max": 5,
    },
    "PeriodicPattern": {
        "steps": [3, 4],
    },
}


def to_cpp_parts(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.rename(columns={"pre_cs": "tmd", "post_cs": "jmd_c"})
        [["tmd", "jmd_c"]]
        .copy()
    )

## 6. Helper functions for OVR type classifiers

For each SP type, a one-vs-rest dataset is built from the SP-only training set.

To reduce class imbalance, negatives are randomly subsampled after the train/test split. The default ratio is `1 : 8` for positive : negative samples, following the original experimental direction.

In [ ]:
def build_ovr_train_set(
    df_train: pd.DataFrame,
    target_type: str,
    neg_pos_ratio: int = OVR_NEG_POS_RATIO,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    df_tmp = df_train.copy()
    df_tmp["ovr_label"] = (df_tmp["sp_type"] == target_type).astype(int)

    df_pos = df_tmp[df_tmp["ovr_label"] == 1].copy()
    df_neg = df_tmp[df_tmp["ovr_label"] == 0].copy()

    n_neg = min(len(df_neg), len(df_pos) * neg_pos_ratio)
    df_neg_sampled = df_neg.sample(n=n_neg, random_state=random_state)

    return (
        pd.concat([df_pos, df_neg_sampled], ignore_index=True)
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )


def build_ovr_test_set(df_test: pd.DataFrame, target_type: str) -> pd.DataFrame:
    df_tmp = df_test.copy()
    df_tmp["ovr_label"] = (df_tmp["sp_type"] == target_type).astype(int)
    return df_tmp.reset_index(drop=True)


def run_cpp_feature_selection(df_train_ovr: pd.DataFrame) -> pd.DataFrame:
    y = df_train_ovr["ovr_label"].values
    df_parts = to_cpp_parts(df_train_ovr)

    cpp = aa.CPP(
        df_scales=df_scales,
        df_parts=df_parts,
        split_kws=SPLIT_KWS,
        accept_gaps=True,
    )

    df_feat = cpp.run(
        labels=y,
        jmd_n_len=0,
        jmd_c_len=POST_CS_LEN,
        split_kws=SPLIT_KWS,
        max_cor=MAX_COR,
    )

    return df_feat


def make_feature_matrix(df_data: pd.DataFrame, df_feat: pd.DataFrame) -> np.ndarray:
    return sf.feature_matrix(
        df_parts=to_cpp_parts(df_data),
        features=df_feat["feature"],
        accept_gaps=True,
    )

## 7. Extract CPP features for each one-vs-rest classifier

In [ ]:
ovr_data = {}
ovr_features = {}

iterator = tqdm(SP_TYPES, desc="CPP OVR feature selection") if tqdm else SP_TYPES

for sp_type in iterator:
    print(f"\n=== CPP features: {sp_type} vs rest ===")

    train_ovr = build_ovr_train_set(train_sp_df, sp_type)
    test_ovr = build_ovr_test_set(test_sp_df, sp_type)

    df_feat_type = run_cpp_feature_selection(train_ovr)

    ovr_data[sp_type] = {
        "train": train_ovr,
        "test": test_ovr,
    }
    ovr_features[sp_type] = df_feat_type

    df_feat_type.to_csv(
        FEATURE_DIR / "metadata" / f"features_ovr_{sp_type}.csv",
        index=False,
    )

    print("train:", train_ovr["ovr_label"].value_counts().to_dict())
    print("test:", test_ovr["ovr_label"].value_counts().to_dict())
    print("n_features:", len(df_feat_type))

## 8. Compare models for each OVR classifier

Each OVR classifier is evaluated over repeated runs. The test set remains the original SP-only test set, while each training set is balanced by negative subsampling.

In [ ]:
def get_ovr_models(seed: int, y_train: np.ndarray):
    pos = int((y_train == 1).sum())
    neg = int((y_train == 0).sum())
    scale_pos_weight = neg / max(pos, 1)

    models = {
        "Logistic Regression": LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            random_state=seed,
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=500,
            class_weight="balanced",
            random_state=seed,
            n_jobs=-1,
        ),
        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=500,
            class_weight="balanced",
            random_state=seed,
            n_jobs=-1,
        ),
    }

    if XGBClassifier is not None:
        models["XGBoost"] = XGBClassifier(
            n_estimators=800,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            scale_pos_weight=scale_pos_weight,
            random_state=seed,
            eval_metric="logloss",
            n_jobs=-1,
        )

    if LGBMClassifier is not None:
        models["LightGBM"] = LGBMClassifier(
            n_estimators=800,
            learning_rate=0.05,
            class_weight="balanced",
            random_state=seed,
            n_jobs=-1,
            verbose=-1,
        )

    if CatBoostClassifier is not None:
        models["CatBoost"] = CatBoostClassifier(
            iterations=800,
            learning_rate=0.05,
            depth=6,
            auto_class_weights="Balanced",
            random_state=seed,
            verbose=False,
        )

    return models


def evaluate_binary_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
    }


ovr_runs = []
ovr_matrices = {}

outer = tqdm(SP_TYPES, desc="OVR model comparison") if tqdm else SP_TYPES

for sp_type in outer:
    train_ovr = ovr_data[sp_type]["train"]
    test_ovr = ovr_data[sp_type]["test"]
    df_feat_type = ovr_features[sp_type]

    X_train_t = make_feature_matrix(train_ovr, df_feat_type)
    y_train_t = train_ovr["ovr_label"].values

    X_test_t = make_feature_matrix(test_ovr, df_feat_type)
    y_test_t = test_ovr["ovr_label"].values

    ovr_matrices[sp_type] = {
        "X_train": X_train_t,
        "y_train": y_train_t,
        "X_test": X_test_t,
        "y_test": y_test_t,
    }

    for run_idx in range(N_RUNS):
        seed = RANDOM_STATE + run_idx
        for model_name, model in get_ovr_models(seed, y_train_t).items():
            metrics = evaluate_binary_model(
                model,
                X_train_t,
                y_train_t,
                X_test_t,
                y_test_t,
            )
            ovr_runs.append(
                {
                    "sp_type": sp_type,
                    "run": run_idx,
                    "model": model_name,
                    **metrics,
                }
            )

ovr_runs_df = pd.DataFrame(ovr_runs)

ovr_summary = (
    ovr_runs_df
    .groupby(["sp_type", "model"])
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        precision_mean=("precision", "mean"),
        precision_std=("precision", "std"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
    )
    .reset_index()
    .sort_values(["sp_type", "f1_mean"], ascending=[True, False])
)

ovr_runs_df.to_csv(METRIC_DIR / "sp_type_ovr_model_runs.csv", index=False)
ovr_summary.to_csv(METRIC_DIR / "sp_type_ovr_model_summary.csv", index=False)

display(ovr_summary)

## 9. Visualize OVR benchmark results

In [ ]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=ovr_summary,
    x="sp_type",
    y="f1_mean",
    hue="model",
)

plt.ylabel("F1 score")
plt.xlabel("SP type")
plt.title(f"OVR SP type classifier comparison ({N_RUNS} runs)")
plt.ylim(0, 1.05)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(PLOT_DIR / "sp_type_ovr_f1_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

best_ovr_rows = (
    ovr_summary
    .sort_values(["sp_type", "f1_mean"], ascending=[True, False])
    .groupby("sp_type", as_index=False)
    .first()
)

display(best_ovr_rows[["sp_type", "model", "f1_mean", "f1_std", "accuracy_mean"]])

## 10. Train final OVR models and predict SP type

The best model for each one-vs-rest task is selected from the repeated benchmark table and retrained on that task's balanced training set.

The final multiclass SP type prediction is obtained by selecting the SP type with the highest one-vs-rest probability.

In [ ]:
def instantiate_model_by_name(model_name: str, seed: int, y_train: np.ndarray):
    return get_ovr_models(seed, y_train)[model_name]


final_ovr_models = {}
ovr_test_proba = pd.DataFrame(index=test_sp_df.index)

for _, row in best_ovr_rows.iterrows():
    sp_type = row["sp_type"]
    model_name = row["model"]

    X_train_t = ovr_matrices[sp_type]["X_train"]
    y_train_t = ovr_matrices[sp_type]["y_train"]
    X_test_t = ovr_matrices[sp_type]["X_test"]

    model = instantiate_model_by_name(model_name, RANDOM_STATE, y_train_t)
    model.fit(X_train_t, y_train_t)

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test_t)[:, 1]
    else:
        proba = model.predict(X_test_t)

    final_ovr_models[sp_type] = {
        "model_name": model_name,
        "model": model,
        "features": ovr_features[sp_type],
    }
    ovr_test_proba[sp_type] = proba

    joblib.dump(
        model,
        MODEL_DIR / f"model_ovr_{sp_type}.pkl",
    )

ovr_test_proba.to_csv(METRIC_DIR / "sp_type_ovr_test_probabilities.csv", index=False)

y_true_type = test_sp_df["sp_type"].values
y_pred_type = ovr_test_proba.idxmax(axis=1).values

print(classification_report(y_true_type, y_pred_type, digits=4))

## 11. Confusion matrix for OVR multiclass prediction

In [ ]:
cm = confusion_matrix(y_true_type, y_pred_type, labels=SP_TYPES)
cm_df = pd.DataFrame(cm, index=SP_TYPES, columns=SP_TYPES)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm_df,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=True,
)
plt.xlabel("Predicted type")
plt.ylabel("True type")
plt.title("SP type prediction — OVR classifiers")
plt.tight_layout()
plt.savefig(PLOT_DIR / "sp_type_ovr_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

## 12. Pairwise refiners

Two difficult type boundaries are refined with pairwise CPP classifiers:

- `LIPO` vs `TATLIPO`
- `SP` vs `TAT`

These refiners are trained only on the relevant pair of SP types and applied only when the OVR prediction falls into the corresponding pair.

In [ ]:
def build_pair_dataset(
    df_train: pd.DataFrame,
    type_negative: str,
    type_positive: str,
    random_state: int = RANDOM_STATE,
):
    df_pair = df_train[df_train["sp_type"].isin([type_negative, type_positive])].copy()
    df_pair["pair_label"] = (df_pair["sp_type"] == type_positive).astype(int)

    df_pos = df_pair[df_pair["pair_label"] == 1]
    df_neg = df_pair[df_pair["pair_label"] == 0]

    # Balance the pairwise training set for rare positives.
    n = min(len(df_pos), len(df_neg))
    df_pos_bal = df_pos.sample(n=n, random_state=random_state) if len(df_pos) > n else df_pos
    df_neg_bal = df_neg.sample(n=n, random_state=random_state) if len(df_neg) > n else df_neg

    df_pair_bal = (
        pd.concat([df_pos_bal, df_neg_bal], ignore_index=True)
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )

    return df_pair_bal


def train_pair_refiner(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    type_negative: str,
    type_positive: str,
):
    print(f"\n=== Pairwise refiner: {type_negative} vs {type_positive} ===")

    train_pair = build_pair_dataset(
        df_train,
        type_negative=type_negative,
        type_positive=type_positive,
    )

    test_pair = df_test[df_test["sp_type"].isin([type_negative, type_positive])].copy()
    test_pair["pair_label"] = (test_pair["sp_type"] == type_positive).astype(int)

    df_feat_pair = run_cpp_feature_selection(
        train_pair.rename(columns={"pair_label": "ovr_label"})
    )

    X_train_pair = make_feature_matrix(train_pair, df_feat_pair)
    y_train_pair = train_pair["pair_label"].values

    X_test_pair = make_feature_matrix(test_pair, df_feat_pair)
    y_test_pair = test_pair["pair_label"].values

    # XGBoost is used for consistency with the thesis-style final model choice.
    if XGBClassifier is not None:
        pos = int((y_train_pair == 1).sum())
        neg = int((y_train_pair == 0).sum())
        model = XGBClassifier(
            n_estimators=800,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            scale_pos_weight=neg / max(pos, 1),
            random_state=RANDOM_STATE,
            eval_metric="logloss",
            n_jobs=-1,
        )
        model_name = "XGBoost"
    else:
        model = RandomForestClassifier(
            n_estimators=500,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        model_name = "Random Forest"

    model.fit(X_train_pair, y_train_pair)
    y_pred_pair = model.predict(X_test_pair)

    print(f"Model: {model_name}")
    print(classification_report(
        y_test_pair,
        y_pred_pair,
        target_names=[type_negative, type_positive],
        digits=4,
        zero_division=0,
    ))

    return {
        "model_name": model_name,
        "model": model,
        "features": df_feat_pair,
        "type_negative": type_negative,
        "type_positive": type_positive,
        "test_pair": test_pair,
    }


refiner_lipo_tatlipo = train_pair_refiner(
    train_sp_df,
    test_sp_df,
    type_negative="LIPO",
    type_positive="TATLIPO",
)

refiner_sp_tat = train_pair_refiner(
    train_sp_df,
    test_sp_df,
    type_negative="SP",
    type_positive="TAT",
)

joblib.dump(refiner_lipo_tatlipo["model"], MODEL_DIR / "model_refiner_lipo_vs_tatlipo.pkl")
joblib.dump(refiner_sp_tat["model"], MODEL_DIR / "model_refiner_sp_vs_tat.pkl")

## 13. Apply pairwise refiners to OVR predictions

In [ ]:
def apply_pair_refiner(
    predictions: np.ndarray,
    df_test: pd.DataFrame,
    refiner: dict,
    candidate_pair: set[str],
) -> np.ndarray:
    updated = predictions.copy()

    mask = pd.Series(predictions).isin(candidate_pair).values
    if mask.sum() == 0:
        return updated

    df_subset = df_test.loc[mask].copy()
    X_subset = make_feature_matrix(df_subset, refiner["features"])

    pred_pair = refiner["model"].predict(X_subset)
    mapped = np.where(
        pred_pair == 1,
        refiner["type_positive"],
        refiner["type_negative"],
    )

    updated[mask] = mapped
    return updated


y_pred_type_refined = y_pred_type.copy()

y_pred_type_refined = apply_pair_refiner(
    y_pred_type_refined,
    test_sp_df,
    refiner_lipo_tatlipo,
    {"LIPO", "TATLIPO"},
)

y_pred_type_refined = apply_pair_refiner(
    y_pred_type_refined,
    test_sp_df,
    refiner_sp_tat,
    {"SP", "TAT"},
)

print(classification_report(y_true_type, y_pred_type_refined, digits=4))

## 14. Confusion matrix after refiners

In [ ]:
cm_refined = confusion_matrix(y_true_type, y_pred_type_refined, labels=SP_TYPES)
cm_refined_df = pd.DataFrame(cm_refined, index=SP_TYPES, columns=SP_TYPES)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm_refined_df,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=True,
)
plt.xlabel("Predicted type")
plt.ylabel("True type")
plt.title("SP type prediction — after pairwise refiners")
plt.tight_layout()
plt.savefig(PLOT_DIR / "sp_type_refined_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

## 15. Final metrics

In [ ]:
final_type_metrics = {
    "ovr_accuracy": accuracy_score(y_true_type, y_pred_type),
    "ovr_macro_f1": f1_score(y_true_type, y_pred_type, average="macro"),
    "ovr_weighted_f1": f1_score(y_true_type, y_pred_type, average="weighted"),
    "refined_accuracy": accuracy_score(y_true_type, y_pred_type_refined),
    "refined_macro_f1": f1_score(y_true_type, y_pred_type_refined, average="macro"),
    "refined_weighted_f1": f1_score(y_true_type, y_pred_type_refined, average="weighted"),
    "n_train_sp": int(len(train_sp_df)),
    "n_test_sp": int(len(test_sp_df)),
    "sp_types": SP_TYPES,
    "min_pre_cs_len": MIN_PRE_CS_LEN,
    "post_cs_len": POST_CS_LEN,
    "ovr_neg_pos_ratio": OVR_NEG_POS_RATIO,
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "n_runs_benchmark": N_RUNS,
}

metrics_path = METRIC_DIR / "sp_type_final_metrics.json"
with open(metrics_path, "w") as file:
    json.dump(final_type_metrics, file, indent=2)

pd.DataFrame({
    "entry": test_sp_df["entry"].values,
    "true_sp_type": y_true_type,
    "ovr_pred_sp_type": y_pred_type,
    "refined_pred_sp_type": y_pred_type_refined,
}).to_csv(METRIC_DIR / "sp_type_test_predictions.csv", index=False)

print(f"Saved final metrics to: {metrics_path}")
final_type_metrics